# Transcripción de las Open Goldberg Variations

Se transcriben las 32 pistas de la grabación de Kimiko Ishizaka de las Variaciones Goldberg (Bach, BWV 988), publicada bajo licencia Creative Commons Zero junto con una edición en musicografía Braille de la misma obra. Esa edición sirve de referencia humana para medir la Exactitud de Símbolo Braille del sistema.

La Variación 25 dura 9.3 minutos y queda fuera del alcance de cinco minutos del sistema, así que no se transcribe.

Cada pista se transcribe dos veces, con el checkpoint publicado de Kong y con el checkpoint del ajuste fino (salida de `amt-finetune`, versión 3). Las dos pasan antes por el normalizador de nivel. El notebook solo guarda las notas; la traducción a Braille y el cálculo del BSA se hacen fuera, con el código del repositorio, a partir de estos JSON.

## 1. Instalación

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("--no-deps", "piano_transcription_inference", "torchlibrosa", "mido")
pip("librosa", "soundfile")

## 2. Grabación

Las pistas se descargan del elemento `The_Open_Goldberg_Variations-11823` de Internet Archive, que las distribuye en MP3 con la misma licencia.

In [ ]:
import json, time, urllib.parse, urllib.request
from pathlib import Path

ITEM = "The_Open_Goldberg_Variations-11823"
AUDIO = Path("/kaggle/working/audio")
AUDIO.mkdir(parents=True, exist_ok=True)
MAX_SECONDS = 300

meta = json.load(urllib.request.urlopen(f"https://archive.org/metadata/{ITEM}"))
tracks = sorted((f for f in meta["files"] if f["name"].endswith(".mp3")), key=lambda f: f["name"])
for i, f in enumerate(tracks):
    f["pieza"] = i
    f["dentro_de_alcance"] = float(f["length"]) <= MAX_SECONDS
    destino = AUDIO / f["name"]
    if f["dentro_de_alcance"] and not destino.exists():
        url = f"https://archive.org/download/{ITEM}/" + urllib.parse.quote(f["name"])
        urllib.request.urlretrieve(url, destino)

for f in tracks:
    print(f"{f['pieza']:2d}  {float(f['length']) / 60:5.2f} min  {'' if f['dentro_de_alcance'] else 'fuera de alcance'}  {f['name'][19:70]}")

## 3. Modelos y normalización

El normalizador es el mismo de `src/amt/audio.py`: quita la componente continua y lleva el RMS de los tramos con sonido a -24.9 dBFS sin recortar picos.

In [ ]:
import glob
import numpy as np
import librosa
import torch
from piano_transcription_inference import PianoTranscription, sample_rate as SR

TARGET_DBFS = -24.9

def db(x):
    return 20.0 * np.log10(max(x, 1e-12))

def active_rms(x, frame=2048):
    n = len(x) // frame
    rms = np.sqrt(np.mean(x[: n * frame].reshape(n, frame) ** 2, axis=1))
    loud = rms[rms >= rms.max() * 10 ** (-40 / 20)]
    return float(np.sqrt(np.mean(loud ** 2)))

def normalize_loudness(x):
    x = np.asarray(x, dtype=np.float32) - np.mean(x)
    level = active_rms(x)
    if db(level) < -80:
        return x
    gain = 10 ** ((TARGET_DBFS - db(level)) / 20)
    gain = min(gain, 0.99 / (np.abs(x).max() + 1e-12))
    return (x * gain).astype(np.float32)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE = str(Path.home() / "piano_transcription_inference_data" / "note_F1=0.9677_pedal_F1=0.9186.pth")
TUNED = glob.glob("/kaggle/input/**/kong_ajustado.pth", recursive=True)
assert TUNED, "Agregar la salida de amt-finetune como input del notebook"
MODELS = {"original": BASE, "ajustado": TUNED[0]}
print(DEVICE, MODELS)

## 4. Transcripción

Un JSON por pieza y modelo con las notas (inicio, fin, altura MIDI y velocidad), los tramos de pedal, la duración del audio y el tiempo de inferencia. Si la sesión se cae, al volver a correr solo se procesa lo que falta.

In [ ]:
OUT = Path("/kaggle/working/goldberg")
OUT.mkdir(exist_ok=True)

for model_name, ckpt in MODELS.items():
    transcriber = PianoTranscription(device=DEVICE, checkpoint_path=ckpt)
    for f in tracks:
        destino = OUT / f"{model_name}__{f['pieza']:02d}.json"
        if not f["dentro_de_alcance"] or destino.exists():
            continue
        audio, _ = librosa.load(str(AUDIO / f["name"]), sr=SR, mono=True)
        audio = normalize_loudness(audio)
        t0 = time.time()
        out = transcriber.transcribe(audio, None)
        latency = time.time() - t0
        notes = [[float(e["onset_time"]), float(e["offset_time"]), int(e["midi_note"]), int(e["velocity"])]
                 for e in out["est_note_events"]]
        pedals = [[float(e["onset_time"]), float(e["offset_time"])] for e in out.get("est_pedal_events", [])]
        destino.write_text(json.dumps({
            "pieza": f["pieza"], "archivo": f["name"], "modelo": model_name,
            "duracion_s": len(audio) / SR, "latencia_s": latency,
            "notas": notes, "pedal": pedals,
        }))
        print(f"{model_name:9s} {f['pieza']:2d}  {len(notes):5d} notas  {latency:5.1f} s")
    del transcriber
    torch.cuda.empty_cache()

## 5. Resumen

In [ ]:
import pandas as pd

rows = [json.loads(p.read_text()) for p in OUT.glob("*.json")]
df = pd.DataFrame([{k: r[k] for k in ("pieza", "modelo", "duracion_s", "latencia_s")} | {"notas": len(r["notas"])} for r in rows])
df["cociente"] = df["latencia_s"] / df["duracion_s"]
df.pivot_table(index="pieza", columns="modelo", values="notas")

In [ ]:
df.groupby("modelo")[["duracion_s", "latencia_s", "cociente", "notas"]].agg(["mean", "max"]).round(3)